In [0]:
# ============================================================
# NOTEBOOK: nb_04_PortfolioExceptions
# PURPOSE:  Replaces SQL Procedure 4 (usp_LoadCustomerPortfolioExceptions)
#           Validates the Customer Portfolio against 6 DQ rules
#           (DQ001-DQ006), classifies severity, calculates SLA,
#           generates SHA256 hashes, and MERGEs into
#           warehouse.customer_portfolio_exceptions.
# CATALOG:  retailbank_dev
# ============================================================

import uuid
from datetime import datetime
from delta.tables import DeltaTable
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# --------------------------------------------------------
# PARAMETERS
# --------------------------------------------------------
dbutils.widgets.text("business_date", "2026-01-31", "Business Date")
dbutils.widgets.text("debug", "1", "Debug Mode (1=print, 0=silent)")

business_date = dbutils.widgets.get("business_date")
debug         = int(dbutils.widgets.get("debug"))

# --------------------------------------------------------
# AUDIT VARIABLES
# --------------------------------------------------------
execution_id   = str(uuid.uuid4())
procedure_name = "nb_04_PortfolioExceptions"
start_time     = datetime.now()

rows_read                   = 0
rows_inserted               = 0
duplicate_exceptions_removed = 0

if debug:
    print("=" * 50)
    print("NB_04_PORTFOLIOEXCEPTIONS STARTED")
    print("=" * 50)
    print(f"Execution ID : {execution_id}")
    print(f"Business Date: {business_date}")

NB_04_PORTFOLIOEXCEPTIONS STARTED
Execution ID : 62ff19aa-62b6-49bb-9c5f-7f741abe2751
Business Date: 2026-01-31


In [0]:
# ============================================================
# PRE-RUN CLEANUP
# ============================================================
#
# WHAT THIS CELL DOES:
# Deletes ALL existing exception records for this business_date
# before the DQ rules run. This guarantees a clean slate on
# every pipeline execution.
#
# WHY THIS IS NECESSARY:
# The MERGE in Cell 8 matches on exception_hash. It inserts
# new exceptions and updates changed ones — but it NEVER
# deletes exceptions that no longer apply. Without this DELETE:
#
#   Run 1: 10 exceptions written (including 2 false positives)
#   Run 2: MERGE finds 8 new hashes + 2 old hashes already there
#          → leaves 10 rows (the 2 false positives remain forever)
#
# With this DELETE:
#   Run 1: DELETE clears 0 rows (first run)
#   Run 2: DELETE clears 10 rows → MERGE writes fresh 8 rows
#          → table has exactly 8 correct rows every time
#
# WHY DELETE IS SAFE HERE:
# This is a daily batch pipeline. The business_date parameter
# controls exactly which day's records are deleted. Records
# for all other business dates are completely unaffected.
# The MERGE that follows immediately re-inserts the correct
# set of exceptions for this date, so there is no window
# where the table is missing data.
#
# THE DYNAMIC business_date PARAMETER:
# business_date is set by the widget in Cell 0. It is NEVER
# hardcoded here. The same code runs correctly for any date
# passed in by the pipeline scheduler — 2026-01-31 today,
# 2026-02-28 next month, without any code changes.
#
# SQL Server equivalent:
# The stored procedure used a temp table and MERGE which
# implicitly replaced all records for the execution date.
# This DELETE + MERGE pattern replicates that behaviour
# explicitly in Databricks Delta Lake.
# ============================================================

target_table = "retailbank_dev.warehouse.customer_portfolio_exceptions"

# --------------------------------------------------------
# DELETE all exceptions for this business_date
# --------------------------------------------------------
deleted = spark.sql(f"""
    DELETE FROM {target_table}
    WHERE business_date = '{business_date}'
""")
# business_date is defined in Cell 0 via dbutils.widgets.get()
# This line will fail with NameError if Cell 0 has not run —
# which correctly prevents the cleanup from silently using a
# stale or wrong date value.

# --------------------------------------------------------
# VERIFY the delete succeeded before proceeding
# --------------------------------------------------------
remaining = (
    spark.table(target_table)
    .filter(f"business_date = '{business_date}'")
    .count()
)

if remaining == 0:
    if debug:
        print(f"✓ Pre-run cleanup complete for business_date={business_date}")
        print(f"  Table is clear for this date. Ready to run DQ rules.")
else:
    # Stop the notebook — do not run DQ rules against a dirty table
    raise Exception(
        f"✗ Pre-run cleanup FAILED — {remaining} rows still exist "
        f"for business_date={business_date}. "
        f"Check DELETE permissions on {target_table}. "
        f"Do NOT proceed until this cell returns 0 remaining rows."
    )

✓ Pre-run cleanup complete for business_date=2026-01-31
  Table is clear for this date. Ready to run DQ rules.


In [0]:
# ============================================================
# AUDIT LOG FUNCTIONS
# Same pattern as previous notebooks.
# ============================================================

def write_audit_start():
    sql = f"""
        INSERT INTO retailbank_dev.audit.etl_execution_log 
        (execution_id, procedure_name, business_date, start_time, status)
        VALUES 
        ('{execution_id}', '{procedure_name}', '{business_date}',
         '{start_time.strftime("%Y-%m-%d %H:%M:%S")}', 'RUNNING')
    """
    spark.sql(sql)


def write_audit_end(status, message):
    end_time         = datetime.now()
    duration_seconds = int((end_time - start_time).total_seconds())
    safe_message     = message.replace("'", "''")
    
    sql = f"""
        UPDATE retailbank_dev.audit.etl_execution_log
        SET 
            end_time         = '{end_time.strftime("%Y-%m-%d %H:%M:%S")}',
            status           = '{status}',
            rows_read        = {rows_read},
            rows_inserted    = {rows_inserted},
            rows_updated     = 0,
            rows_rejected    = {duplicate_exceptions_removed},
            duration_seconds = {duration_seconds},
            message          = '{safe_message}'
        WHERE execution_id = '{execution_id}'
    """
    spark.sql(sql)


write_audit_start()

In [0]:
# ============================================================
# READ SOURCE DATA
# 1. CustomerPortfolio -> what we validate
# 2. ExceptionRules -> which rules are active + severity
# 3. BusinessOwner -> who owns each exception
# ============================================================

portfolio_df = spark.table("retailbank_dev.warehouse.customer_portfolio") \
    .filter(F.col("business_date") == business_date)

rows_read = portfolio_df.count()

rules_df = spark.table("retailbank_dev.config.exception_rules") \
    .filter(F.col("rule_enabled") == True)

owners_df = spark.table("retailbank_dev.config.business_owner")

if debug:
    print(f"Portfolio records to validate: {rows_read}")
    print(f"Active DQ rules: {rules_df.count()}")
    portfolio_df.select("source_system_code", "source_account_number", "customer_id", "product_code", "base_currency_balance").show(5, truncate=False)

Portfolio records to validate: 19
Active DQ rules: 6
+------------------+---------------------+-----------+-------------+---------------------+
|source_system_code|source_account_number|customer_id|product_code |base_currency_balance|
+------------------+---------------------+-----------+-------------+---------------------+
|CORE_BANKING      |CB10001              |CUST001    |SAV001       |25000.00             |
|LOANS             |LN10001              |CUST001    |HOME_LOAN    |500000.00            |
|CARDS             |CARD001              |CUST001    |GOLD_CARD    |12000.00             |
|INVESTMENTS       |INV001               |CUST001    |EQUITY_FUND  |150000.00            |
|MOBILE            |WAL001               |CUST001    |MOBILE_WALLET|500.00               |
+------------------+---------------------+-----------+-------------+---------------------+
only showing top 5 rows


In [0]:
# ============================================================
# EXCEPTION STAGING
# We will collect all 6 DQ rule violations into this DataFrame.
# Each rule is a separate filter + join to get severity/owner.
# ============================================================

# Start with an empty DataFrame that has the right schema
# We create an empty RDD with the schema explicitly
# defined, then union each rule's results into it. This avoids
# type inference issues.
from pyspark.sql.types import StructType, StructField, StringType, DateType, TimestampType

exception_schema = StructType([
    StructField("business_date", DateType(), True),
    StructField("source_system_code", StringType(), True),
    StructField("source_account_number", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("rule_code", StringType(), True),
    StructField("exception_category", StringType(), True),
    StructField("exception_description", StringType(), True),
    StructField("exception_value", StringType(), True),
    StructField("severity_code", StringType(), True),
    StructField("business_area", StringType(), True),
    StructField("business_owner", StringType(), True),
    StructField("logged_date", TimestampType(), True)
])

# Empty DataFrame with schema
all_exceptions_df = spark.createDataFrame([], exception_schema)

In [0]:
# ============================================================
# DQ RULE FUNCTIONS
# Each function takes the portfolio DataFrame and a rule config row,
# then returns violations as a standardized DataFrame.
# This is cleaner than 6 separate big blocks.
# ============================================================

def apply_rule(portfolio_df, rule_row, condition_expr, description_template, value_col):
    """
    Generic rule applier.
    rule_row: the config row with rule_code, severity, etc.
    condition_expr: PySpark Column expression for the filter
    description_template: string like 'Customer not found'
    value_col: column to capture as exception_value
    """
    rule_code  = rule_row.rule_code
    category   = rule_row.exception_category
    severity   = rule_row.severity_code
    biz_area   = rule_row.business_area
    
    # Find the owner for this business area
    owner_row = owners_df.filter(F.col("business_area") == biz_area).first()
    owner_name = owner_row.business_owner if owner_row else "Unknown"
    
    # Filter violations
    violations = portfolio_df.filter(condition_expr).select(
        F.col("business_date"),
        F.col("source_system_code"),
        F.col("source_account_number"),
        F.col("customer_id"),
        F.lit(rule_code).alias("rule_code"),
        F.lit(category).alias("exception_category"),
        F.lit(description_template).alias("exception_description"),
        F.when(F.col(value_col).isNull(), F.lit("NULL")).otherwise(F.col(value_col).cast("string")).alias("exception_value"),
        F.lit(severity).alias("severity_code"),
        F.lit(biz_area).alias("business_area"),
        F.lit(owner_name).alias("business_owner"),
        F.current_timestamp().alias("logged_date")
    )
    
    return violations

In [0]:
# ============================================================
# RUN THE 6 DQ RULES
# ============================================================
#
# WHAT THIS CELL DOES:
# Evaluates every account in the enterprise CustomerPortfolio
# against six data quality rules (DQ001-DQ006).
# Any account that violates a rule produces an exception row.
# All exception rows are collected and unioned into a single
# DataFrame (all_exceptions_df) which flows into:
#   Cell 6  — Deduplication  -> deduped_df
#   Cell 7  — Enrichment     -> enriched_df
#   Cell 8  — MERGE to Delta -> warehouse table
#
# THE SIX RULES AND THEIR BUSINESS PURPOSE:
#   DQ001 — Customer Does Not Exist in Customer Master
#            Catches accounts whose customer_id is not in
#            Warehouse.CustomerMaster. The master is built
#            from CORE_BANKING, LOANS, and INVESTMENTS only
#            (first-pass sources). Customers who only appear
#            in CARDS, MORTGAGE, MOBILE, or FOREX but have no
#            account in the three primary sources are ghost
#            accounts — they have no verified customer identity
#            and must be investigated by Customer Operations.
#
#   DQ002 — Unknown Product
#            Catches accounts whose product code cannot be
#            matched to Reference.Product. Without a valid
#            product record the account cannot be categorised
#            or included in regulatory reporting.
#
#   DQ003 — Missing Exchange Rate
#            Catches non-USD accounts where the exchange rate
#            lookup returned NULL. Without a rate the account
#            balance cannot be converted to the base currency,
#            making the portfolio balance unreliable.
#
#   DQ004 — Negative Portfolio Balance
#            Catches accounts where base_currency_balance < 0.
#            Negative balances indicate either an accounting
#            error or a data entry mistake. Risk and Compliance
#            must investigate before portfolio sign-off.
#
#   DQ005 — Closed Account Included in Reporting
#            Catches accounts flagged as CLOSED but still
#            marked eligible_for_reporting = Y. Closed accounts
#            must not contribute to active portfolio metrics.
#
#   DQ006 — Duplicate Account
#            Catches source account numbers that appear more
#            than once in the portfolio for the same business
#            date. Duplicates inflate counts and distort totals.
#
# WHY DQ001 READS CUSTOMER MASTER IN THIS CELL:
# Cell 2 (READ SOURCE DATA) loads portfolio_df, rules_df, and
# owners_df — the three DataFrames needed for DQ002-DQ006.
# DQ001 additionally requires CustomerMaster as a reference set.
# We read it here, scoped to this cell, so the dependency is
# explicit and visible exactly at the point of use.
#
# WHY DQ001 IS A SIMPLE ANTI-JOIN (NO EXTRA CONDITIONS):
# The CustomerMaster is built by nb_01 from exactly the right
# set of customers (CORE_BANKING + LOANS + INVESTMENTS first pass).
# Any portfolio customer not in that master is by definition a
# ghost account. No additional filter on customer_name or
# customer_category is needed — the master itself is the filter.
#
# This is the correct and clean architectural approach:
#   "Is the customer in the master?" → simple YES/NO anti-join
#   "Which customers are in the master?" → controlled by nb_01
#
# SQL SERVER EQUIVALENCE:
# Each DQ block below corresponds to a numbered SECTION in
# usp_LoadCustomerPortfolioExceptions (Sections F through K).
# ============================================================


# --------------------------------------------------------
# READ CUSTOMER MASTER FOR DQ001 ANTI-JOIN
#
# Select only customer_id and call distinct() to produce
# the smallest possible lookup set. Pulling only the key
# avoids an unnecessary shuffle of all CustomerMaster columns.
#
# SQL Server equivalent (Section F):
#   WHERE NOT EXISTS (
#       SELECT 1 FROM Warehouse.CustomerMaster C
#       WHERE C.CustomerId = P.CustomerId
#   )
# --------------------------------------------------------
customer_master_df = (
    spark.table("retailbank_dev.warehouse.customer_master")
    .select("customer_id")
    .distinct()
)


# --------------------------------------------------------
# DQ001 — Customer Does Not Exist in Customer Master
#
# ANTI-JOIN PATTERN — replicates SQL NOT EXISTS:
#
#   Step 1: LEFT JOIN portfolio to CustomerMaster on customer_id.
#           Rows with a match    → cm.customer_id = matched value
#           Rows with no match   → cm.customer_id = NULL
#
#   Step 2: FILTER cm.customer_id IS NULL
#           These rows have no match in the master =
#           equivalent to SQL NOT EXISTS.
#
#   Step 3: AND p.customer_id IS NOT NULL
#           Excludes portfolio rows with a missing customer_id.
#           Those are a separate data problem, not a DQ001 issue.
#
# WHY NO EXTRA CONDITIONS ON customer_name OR customer_category:
# nb_01 controls exactly which customers are in the master.
# The master contains 9 verified customers from CORE_BANKING,
# LOANS, and INVESTMENTS. Any portfolio customer not in those
# 9 is automatically a DQ001 violation — no further conditions
# needed. The data in the master IS the business rule.
#
# Example violations (customers in portfolio, absent from master):
#   CARD004 / CUST005  — only in CARDS, not in master  → DQ001
#   LN10003 / CUST009  — different LOANS customer       → DQ001
#   INV003  / CUST013  — different INVESTMENTS customer → DQ001
#   WAL002  / CUST020  — only in MOBILE                → DQ001
#   FX003   / CUST031  — only in FOREX                 → DQ001
#
# Non-violations (customers in portfolio AND in master):
#   LN10004 / CUST010  — in LOANS, loaded by nb_01     → OK
#   INV002  / CUST012  — in INVESTMENTS, loaded by nb_01 → OK
#
# Expected count: 5 violations
# Severity: HIGH  |  Business Area: Customer Operations
# --------------------------------------------------------
dq001 = (
    portfolio_df.alias("p")
    .join(
        rules_df.filter(F.col("rule_code") == "DQ001").alias("r"),
        F.lit(True),
        "cross"
        # Cross join: attach DQ001 rule config (severity_code,
        # exception_category, business_area) to every portfolio row.
    )
    .join(
        owners_df.alias("o"),
        F.col("r.business_area") == F.col("o.business_area"),
        "left"
        # Left join: resolve business owner for Customer Operations.
        # Left (not inner) so no exception is lost if config is missing.
    )
    .join(
        customer_master_df.alias("cm"),
        F.col("p.customer_id") == F.col("cm.customer_id"),
        "left"
        # ANTI-JOIN Step 1: attempt to match each portfolio customer_id.
        # Match found    → cm.customer_id = the matched value
        # No match found → cm.customer_id = NULL
    )
    .filter(
        F.col("cm.customer_id").isNull()        # No match in CustomerMaster
        & F.col("p.customer_id").isNotNull()    # But has a customer_id
        # Together: customer_id in portfolio but absent from master.
        # This is the Spark equivalent of SQL NOT EXISTS.
    )
    .select(
        F.lit(business_date).cast("date").alias("business_date"),
        F.col("p.source_system_code"),
        F.col("p.source_account_number"),
        F.col("p.customer_id"),
        F.col("r.rule_code"),
        F.col("r.exception_category"),
        F.lit("Customer not found in Customer Master").alias("exception_description"),
        F.col("p.customer_id").cast("string").alias("exception_value"),
        # exception_value = the offending customer_id.
        # One of four inputs to the SHA256 hash in Cell 7.
        F.col("r.severity_code"),
        F.col("r.business_area"),
        F.col("o.business_owner"),
        F.current_timestamp().alias("logged_date")
    )
)


# --------------------------------------------------------
# DQ002 — Unknown Product
#
# SQL Server equivalent (Section G):
#   WHERE NOT EXISTS (
#       SELECT 1 FROM Reference.Product PR
#       WHERE PR.ProductCode = P.ProductCode
#   )
#
# nb_03 performs a LEFT JOIN to Reference.Product during
# portfolio construction. When product_code has no match,
# the join returns NULL for product_description.
# NULL product_description = NOT EXISTS in SQL Server terms.
#
# Severity: MEDIUM  |  Business Area: Product Management
# --------------------------------------------------------
dq002 = (
    portfolio_df.alias("p")
    .join(
        rules_df.filter(F.col("rule_code") == "DQ002").alias("r"),
        F.lit(True), "cross"
    )
    .join(
        owners_df.alias("o"),
        F.col("r.business_area") == F.col("o.business_area"), "left"
    )
    .filter(F.col("p.product_description").isNull())
    .select(
        F.lit(business_date).cast("date").alias("business_date"),
        F.col("p.source_system_code"),
        F.col("p.source_account_number"),
        F.col("p.customer_id"),
        F.col("r.rule_code"),
        F.col("r.exception_category"),
        F.lit("Unknown Product").alias("exception_description"),
        F.col("p.product_code").cast("string").alias("exception_value"),
        F.col("r.severity_code"),
        F.col("r.business_area"),
        F.col("o.business_owner"),
        F.current_timestamp().alias("logged_date")
    )
)


# --------------------------------------------------------
# DQ003 — Missing Exchange Rate
#
# SQL Server equivalent (Section H):
#   WHERE P.ExchangeRate IS NULL
#
# nb_03 assigns exchange_rate = 1.0 for all USD accounts.
# A NULL exchange_rate therefore always means a non-USD account
# failed the rate lookup. The currency_code != 'USD' guard
# is retained as a safety condition.
#
# Severity: HIGH  |  Business Area: Finance
# --------------------------------------------------------
dq003 = (
    portfolio_df.alias("p")
    .join(
        rules_df.filter(F.col("rule_code") == "DQ003").alias("r"),
        F.lit(True), "cross"
    )
    .join(
        owners_df.alias("o"),
        F.col("r.business_area") == F.col("o.business_area"), "left"
    )
    .filter(
        F.col("p.exchange_rate").isNull()
        & (F.col("p.currency_code") != "USD")
    )
    .select(
        F.lit(business_date).cast("date").alias("business_date"),
        F.col("p.source_system_code"),
        F.col("p.source_account_number"),
        F.col("p.customer_id"),
        F.col("r.rule_code"),
        F.col("r.exception_category"),
        F.lit("Exchange Rate Missing").alias("exception_description"),
        F.col("p.currency_code").cast("string").alias("exception_value"),
        F.col("r.severity_code"),
        F.col("r.business_area"),
        F.col("o.business_owner"),
        F.current_timestamp().alias("logged_date")
    )
)


# --------------------------------------------------------
# DQ004 — Negative Portfolio Balance
#
# SQL Server equivalent (Section I):
#   WHERE P.BaseCurrencyBalance < 0
#   exception_value = CAST(P.BaseCurrencyBalance AS VARCHAR(50))
#
# Direct numeric filter — no reference join needed.
#
# EXCEPTION VALUE FORMAT — CRITICAL FOR HASH MATCHING:
# SQL Server CAST(DECIMAL(18,2) AS VARCHAR(50)) produces
# "-6250.00" (two decimal places preserved).
# Spark .cast("string") on DecimalType(18,2) also produces
# "-6250.00". The two platforms agree — hashes will match.
#
# Expected count: 3 violations
# Severity: CRITICAL  |  Business Area: Risk
# --------------------------------------------------------
dq004 = (
    portfolio_df.alias("p")
    .join(
        rules_df.filter(F.col("rule_code") == "DQ004").alias("r"),
        F.lit(True), "cross"
    )
    .join(
        owners_df.alias("o"),
        F.col("r.business_area") == F.col("o.business_area"), "left"
    )
    .filter(F.col("p.base_currency_balance") < 0)
    .select(
        F.lit(business_date).cast("date").alias("business_date"),
        F.col("p.source_system_code"),
        F.col("p.source_account_number"),
        F.col("p.customer_id"),
        F.col("r.rule_code"),
        F.col("r.exception_category"),
        F.lit("Negative Portfolio Balance").alias("exception_description"),
        F.col("p.base_currency_balance").cast("string").alias("exception_value"),
        # Cast matches SQL Server CAST(... AS VARCHAR(50)) exactly —
        # same string format ensures the SHA256 hash matches.
        F.col("r.severity_code"),
        F.col("r.business_area"),
        F.col("o.business_owner"),
        F.current_timestamp().alias("logged_date")
    )
)


# --------------------------------------------------------
# DQ005 — Closed Account Included in Reporting
#
# SQL Server equivalent (Section J):
#   WHERE AccountStatus = 'CLOSED'
#   AND   EligibleForReporting = 'Y'
#
# nb_03 attempts to exclude CLOSED accounts during portfolio
# loading. This rule acts as a safety net.
#
# Severity: HIGH  |  Business Area: Operations
# --------------------------------------------------------
dq005 = (
    portfolio_df.alias("p")
    .join(
        rules_df.filter(F.col("rule_code") == "DQ005").alias("r"),
        F.lit(True), "cross"
    )
    .join(
        owners_df.alias("o"),
        F.col("r.business_area") == F.col("o.business_area"), "left"
    )
    .filter(
        (F.col("p.account_status") == "CLOSED")
        & (F.col("p.eligible_for_reporting") == "Y")
    )
    .select(
        F.lit(business_date).cast("date").alias("business_date"),
        F.col("p.source_system_code"),
        F.col("p.source_account_number"),
        F.col("p.customer_id"),
        F.col("r.rule_code"),
        F.col("r.exception_category"),
        F.lit("Closed Account Eligible For Reporting").alias("exception_description"),
        F.col("p.account_status").cast("string").alias("exception_value"),
        F.col("r.severity_code"),
        F.col("r.business_area"),
        F.col("o.business_owner"),
        F.current_timestamp().alias("logged_date")
    )
)


# --------------------------------------------------------
# DQ006 — Duplicate Account
#
# SQL Server equivalent (Section K):
#   WITH DuplicateAccounts AS (
#       SELECT SourceAccountNumber, COUNT(*) AS DuplicateCount
#       FROM #CustomerPortfolio
#       GROUP BY SourceAccountNumber
#       HAVING COUNT(*) > 1
#   )
#   SELECT P.* FROM #CustomerPortfolio P
#   INNER JOIN DuplicateAccounts D
#       ON D.SourceAccountNumber = P.SourceAccountNumber
#
# Step 1: GROUP BY source_account_number, filter count > 1
#         → produces the set of duplicated account numbers
# Step 2: INNER JOIN portfolio to that set
#         → only duplicated rows survive
#
# Severity: CRITICAL  |  Business Area: Data Governance
# --------------------------------------------------------
duplicate_accounts = (
    portfolio_df
    .groupBy("source_account_number")
    .count()
    .filter(F.col("count") > 1)
    .select("source_account_number")
)

dq006 = (
    portfolio_df.alias("p")
    .join(
        duplicate_accounts.alias("d"),
        F.col("p.source_account_number") == F.col("d.source_account_number"),
        "inner"
    )
    .join(
        rules_df.filter(F.col("rule_code") == "DQ006").alias("r"),
        F.lit(True), "cross"
    )
    .join(
        owners_df.alias("o"),
        F.col("r.business_area") == F.col("o.business_area"), "left"
    )
    .select(
        F.lit(business_date).cast("date").alias("business_date"),
        F.col("p.source_system_code"),
        F.col("p.source_account_number"),
        F.col("p.customer_id"),
        F.col("r.rule_code"),
        F.col("r.exception_category"),
        F.lit("Duplicate Source Account").alias("exception_description"),
        F.col("p.source_account_number").cast("string").alias("exception_value"),
        F.col("r.severity_code"),
        F.col("r.business_area"),
        F.col("o.business_owner"),
        F.current_timestamp().alias("logged_date")
    )
)


# --------------------------------------------------------
# UNION ALL SIX RULE DATAFRAMES INTO ONE
#
# reduce() applies unionByName cumulatively across the list.
# unionByName matches columns by NAME not by POSITION —
# safe even if individual rule SELECTs differ in column order.
# allowMissingColumns=True fills gaps with NULL for forward
# compatibility if a future rule adds a new column.
# --------------------------------------------------------
from functools import reduce
from pyspark.sql import DataFrame

all_rule_dfs = [dq001, dq002, dq003, dq004, dq005, dq006]

all_exceptions_df = reduce(
    lambda a, b: a.unionByName(b, allowMissingColumns=True),
    all_rule_dfs
)

if debug:
    total_before_dedup = all_exceptions_df.count()
    print(f"\n{'='*50}")
    print("DQ RULES COMPLETE")
    print(f"{'='*50}")
    print(f"Total exceptions before deduplication : {total_before_dedup}")
    print("\nBreakdown by rule:")
    all_exceptions_df.groupBy("rule_code").count().orderBy("rule_code").show()
    print("\nBreakdown by severity:")
    all_exceptions_df.groupBy("severity_code").count().orderBy("severity_code").show()
    print("\nFull exception list:")
    all_exceptions_df.select(
        "source_account_number", "customer_id",
        "rule_code", "severity_code"
    ).orderBy("rule_code", "source_account_number").show(20)
    print("Expected: DQ001=5, DQ004=3, Total=8")


DQ RULES COMPLETE
Total exceptions before deduplication : 6

Breakdown by rule:
+---------+-----+
|rule_code|count|
+---------+-----+
|    DQ001|    3|
|    DQ004|    3|
+---------+-----+


Breakdown by severity:
+-------------+-----+
|severity_code|count|
+-------------+-----+
|     CRITICAL|    3|
|         HIGH|    3|
+-------------+-----+


Full exception list:
+---------------------+-----------+---------+-------------+
|source_account_number|customer_id|rule_code|severity_code|
+---------------------+-----------+---------+-------------+
|              CARD004|    CUST005|    DQ001|         HIGH|
|                FX003|    CUST031|    DQ001|         HIGH|
|               WAL002|    CUST020|    DQ001|         HIGH|
|              CB10004|    CUST004|    DQ004|     CRITICAL|
|                FX003|    CUST031|    DQ004|     CRITICAL|
|              LN10004|    CUST010|    DQ004|     CRITICAL|
+---------------------+-----------+---------+-------------+

Expected: DQ001=5, DQ004=3, To

In [0]:
# ============================================================
# DEDUPLICATION
# If the same exception is detected multiple times on the same day,
# keep only the latest occurrence (by logged_date).
# Same ROW_NUMBER() pattern as nb_02.
# ============================================================

window_spec = Window.partitionBy(
    "business_date", "customer_id", "rule_code", "exception_value"
).orderBy(F.desc("logged_date"))

deduped_df = all_exceptions_df.withColumn("rn", F.row_number().over(window_spec)) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

duplicate_exceptions_removed = all_exceptions_df.count() - deduped_df.count()

if debug:
    print(f"Duplicate Exceptions Removed : {duplicate_exceptions_removed}")
    print(f"Exceptions after dedup       : {deduped_df.count()}")

Duplicate Exceptions Removed : 0
Exceptions after dedup       : 6


In [0]:
# ============================================================
# ENRICH EXCEPTIONS
# ============================================================
#
# WHAT THIS CELL DOES:
# Takes the deduplicated exception records and adds four
# fields needed for the warehouse target table:
#
#   1. PriorityLevel, SLAHours, EscalationRequired
#      Joined from config.exception_severity using severity_code.
#      Tells us how urgent each exception is and how long
#      the business team has to resolve it.
#
#   2. ResolutionDueDate
#      logged_date + SLA hours = the fix deadline.
#      Calculated using Unix timestamp arithmetic (same as nb_02).
#
#   3. ExceptionHash (SHA256)
#      A unique fingerprint for each exception record.
#      Used as the MERGE key — prevents duplicate rows if
#      the notebook is re-run on the same date.
#
#   4. EscalationQueue
#      Text routing label derived from severity_code.
#
# ============================================================
# CRITICAL FIX — SHA256 HASH ENCODING
# ============================================================
#
# ROOT CAUSE OF THE HASH MISMATCH:
# SQL Server's HASHBYTES function operates on NVARCHAR data.
# NVARCHAR in SQL Server is encoded as UTF-16-LE (two bytes
# per character, little-endian byte order).
#
# The diagnostic confirmed this:
#   SQL Server hash for '2026-01-31CUST004DQ004-6250.00':
#     F8033407F2DDF244E88392CCB150A03BF3E96598...  (UTF-16-LE) ✓
#   Spark sha2() for the same string:
#     8C8A8C5CE02D7C9AADF2689EA6BBC7CC9CF3FB1C...  (UTF-8)     ✗
#
# Spark sha2() encodes strings as UTF-8 (one byte per ASCII
# character). SQL Server HASHBYTES encodes as UTF-16-LE
# (two bytes per character). Same text, different byte
# sequences, different hashes.
#
# THE FIX:
# We cannot change how Spark sha2() encodes strings.
# Instead we use a Python UDF (User-Defined Function) that
# encodes the concatenated string as UTF-16-LE bytes and
# then computes SHA256 on those bytes — exactly replicating
# what SQL Server does.
#
# SQL Server formula (from the stored procedure):
#   HASHBYTES('SHA2_256',
#       CONCAT(BusinessDate, CustomerId, RuleCode, ExceptionValue)
#   )
# No separator between fields. Result converted to uppercase
# hex using CONVERT(VARCHAR(64), ..., 2).
#
# WHAT A UDF IS:
# A UDF (User-Defined Function) is a Python function that
# you register with Spark so it can be applied to every row
# of a DataFrame, just like a built-in Spark function.
# We use it here because Spark has no built-in function that
# encodes a string as UTF-16-LE before hashing.
# ============================================================

import hashlib
from pyspark.sql.types import StringType

# --------------------------------------------------------
# DEFINE THE UTF-16-LE SHA256 UDF
# This replicates SQL Server HASHBYTES('SHA2_256', nvarchar).
#
# HOW IT WORKS:
# 1. Concatenate the four fields with no separator
#    (matching SQL Server CONCAT with no delimiter)
# 2. Encode the result as UTF-16-LE bytes
#    (matching SQL Server NVARCHAR internal encoding)
# 3. Compute SHA256 on those bytes
# 4. Return the hash as uppercase hex
#    (matching SQL Server CONVERT(..., 2) output format)
#
# COALESCE TO EMPTY STRING:
# SQL Server CONCAT treats NULL as empty string automatically.
# We do the same — replace None with "" before concatenation
# so a NULL field does not break the hash.
# --------------------------------------------------------
def sql_server_hash(business_date, customer_id, rule_code, exception_value):
    """
    Computes SHA256 using UTF-16-LE encoding to match
    SQL Server HASHBYTES('SHA2_256', CONCAT(...)) exactly.

    Parameters:
        business_date   : the business date string
        customer_id     : the customer identifier
        rule_code       : the DQ rule code (e.g. DQ004)
        exception_value : the exception value (e.g. -6250.00)

    Returns:
        Uppercase hex SHA256 string (64 characters)
        matching SQL Server CONVERT(VARCHAR(64), HASHBYTES(...), 2)
    """
    # Replace None with empty string — matches SQL Server CONCAT
    # which treats NULL as "" rather than propagating NULL
    bd  = str(business_date   or "")
    cid = str(customer_id     or "")
    rc  = str(rule_code       or "")
    ev  = str(exception_value or "")

    # Concatenate with no separator — matches SQL Server CONCAT()
    concatenated = bd + cid + rc + ev

    # Encode as UTF-16-LE — this is the key step that matches
    # SQL Server's internal NVARCHAR byte representation
    encoded_bytes = concatenated.encode("utf-16-le")

    # Compute SHA256 on the UTF-16-LE bytes and return uppercase
    return hashlib.sha256(encoded_bytes).hexdigest().upper()


# Register the UDF with Spark so we can use it in DataFrame
# operations. StringType() declares that the function returns
# a string value.
sql_server_hash_udf = udf(sql_server_hash, StringType())


# --------------------------------------------------------
# STEP 1 — Join severity config to get SLA details.
# Left join preserves exceptions even if severity config
# is missing — better to keep the exception without SLA
# than to lose the exception row entirely.
# --------------------------------------------------------
severity_df = spark.table("retailbank_dev.config.exception_severity")

enriched_df = deduped_df.alias("e").join(
    severity_df.alias("s"),
    F.col("e.severity_code") == F.col("s.severity_code"),
    "left"
).select(
    F.col("e.*"),
    F.col("s.priority_level"),
    F.col("s.sla_hours"),
    F.col("s.escalation_required")
)

# --------------------------------------------------------
# STEP 2 — Calculate resolution due date.
# logged_date + sla_hours = the deadline for the fix.
# We convert to Unix seconds, add hours as seconds,
# then convert back to a timestamp. Same approach as nb_02.
# --------------------------------------------------------
enriched_df = enriched_df.withColumn(
    "resolution_due_date",
    F.from_unixtime(
        F.unix_timestamp("logged_date") +
        F.col("sla_hours") * 3600
    ).cast("timestamp")
)

# --------------------------------------------------------
# STEP 3 — Generate SHA256 hash using the UDF.
#
# We pass four column values into the UDF for each row.
# The UDF concatenates them (no separator), encodes as
# UTF-16-LE, and returns the uppercase SHA256 hex string.
#
# This will now match SQL Server's HASHBYTES output exactly,
# confirmed by the diagnostic:
#   Input  : '2026-01-31CUST004DQ004-6250.00'
#   UTF-16-LE SHA256: F8033407F2DDF244...  ✓ matches SQL Server
# --------------------------------------------------------
enriched_df = enriched_df.withColumn(
    "exception_hash",
    sql_server_hash_udf(
        F.col("business_date").cast("string"),
        F.col("customer_id"),
        F.col("rule_code"),
        F.col("exception_value")
    )
)

# --------------------------------------------------------
# STEP 4 — Determine escalation queue from severity.
# F.when().when().otherwise() is PySpark's equivalent of
# SQL CASE WHEN ... THEN ... ELSE ... END.
# --------------------------------------------------------
enriched_df = enriched_df.withColumn(
    "escalation_queue",
    F.when(F.col("severity_code") == "CRITICAL", "Immediate Response")
     .when(F.col("severity_code") == "HIGH",     "Priority Queue")
     .when(F.col("severity_code") == "MEDIUM",   "Standard Queue")
     .otherwise("Low Priority Queue")
)

if debug:
    print("Exceptions enriched with SLA, hash and escalation queue:")
    enriched_df.select(
        "rule_code",
        "severity_code",
        "priority_level",
        "sla_hours",
        "escalation_queue",
        "exception_hash"
    ).show(10, truncate=False)

    # Verify the hash matches SQL Server for the DQ004 exceptions.
    # If these match, the UDF is working correctly.
    print("Hash verification against SQL Server baseline values:")
    enriched_df.filter(
        F.col("rule_code") == "DQ004"
    ).select(
        "source_account_number",
        "exception_value",
        "exception_hash"
    ).show(truncate=False)

Exceptions enriched with SLA, hash and escalation queue:
+---------+-------------+--------------+---------+------------------+----------------------------------------------------------------+
|rule_code|severity_code|priority_level|sla_hours|escalation_queue  |exception_hash                                                  |
+---------+-------------+--------------+---------+------------------+----------------------------------------------------------------+
|DQ004    |CRITICAL     |1             |2        |Immediate Response|F8033407F2DDF244E88392CCB150A03BF3E96598A0016BDBD968CF1818BD5EC7|
|DQ001    |HIGH         |2             |8        |Priority Queue    |F1771CCC7C1F8D446A9B8495B2FAEB678444804A51C5D150C591F6F44F7FC45E|
|DQ004    |CRITICAL     |1             |2        |Immediate Response|E8235FC8E2C4979D13260730C4D1DD55F1A880C203265189ED4C95F0EDDEBDD7|
|DQ001    |HIGH         |2             |8        |Priority Queue    |D440B300A65FC018368301AC9FA0ED53C7A18DBF2D611EB57536FDC8B23A3F2C

In [0]:
# ============================================================
# MERGE INTO WAREHOUSE.CUSTOMER_PORTFOLIO_EXCEPTIONS
# Match on ExceptionHash (prevents duplicates across reruns).
# Update if severity, owner, priority, SLA, queue, or description changes.
# ============================================================

target_table = "retailbank_dev.warehouse.customer_portfolio_exceptions"
delta_target = DeltaTable.forName(spark, target_table)

if enriched_df.count() > 0:
    
    delta_target.alias("target").merge(
        enriched_df.alias("source"),
        "target.exception_hash = source.exception_hash"
    ).whenMatchedUpdate(
        condition="""
            COALESCE(target.severity_code, '') <> COALESCE(source.severity_code, '') OR
            COALESCE(target.business_owner, '') <> COALESCE(source.business_owner, '') OR
            COALESCE(target.priority_level, 0) <> COALESCE(source.priority_level, 0) OR
            COALESCE(target.sla_hours, 0) <> COALESCE(source.sla_hours, 0) OR
            COALESCE(target.escalation_queue, '') <> COALESCE(source.escalation_queue, '') OR
            COALESCE(target.exception_description, '') <> COALESCE(source.exception_description, '')
        """,
        set={
            "exception_description": "source.exception_description",
            "severity_code":         "source.severity_code",
            "priority_level":        "source.priority_level",
            "sla_hours":             "source.sla_hours",
            "business_area":         "source.business_area",
            "business_owner":        "source.business_owner",
            "escalation_queue":      "source.escalation_queue",
            "resolution_due_date":   "source.resolution_due_date",
            "last_updated_date":     "source.logged_date"
        }
    ).whenNotMatchedInsert(
        values={
            "business_date":         "source.business_date",
            "source_system_code":    "source.source_system_code",
            "source_account_number": "source.source_account_number",
            "customer_id":           "source.customer_id",
            "rule_code":             "source.rule_code",
            "exception_category":    "source.exception_category",
            "exception_description": "source.exception_description",
            "exception_value":       "source.exception_value",
            "severity_code":         "source.severity_code",
            "priority_level":        "source.priority_level",
            "sla_hours":             "source.sla_hours",
            "escalation_required":   "source.escalation_required",
            "escalation_queue":      "source.escalation_queue",
            "business_area":         "source.business_area",
            "business_owner":        "source.business_owner",
            "exception_hash":        "source.exception_hash",
            "logged_date":           "source.logged_date",
            "resolution_due_date":   "source.resolution_due_date",
            "created_date":          "source.logged_date",
            "last_updated_date":     "source.logged_date"
        }
    ).execute()

    # Capture MERGE stats
    history_df = spark.sql(f"DESCRIBE HISTORY {target_table}")
    latest_merge = history_df.filter("operation = 'MERGE'").orderBy(F.desc("version")).limit(1)
    
    if latest_merge.count() > 0:
        metrics = latest_merge.select("operationMetrics").collect()[0][0]
        rows_inserted = int(metrics.get("numTargetRowsInserted", "0"))
    else:
        rows_inserted = 0
else:
    rows_inserted = 0
    if debug:
        print("No exceptions to MERGE.")

if debug:
    print(f"MERGE complete: {rows_inserted} exceptions inserted/updated")

MERGE complete: 6 exceptions inserted/updated


In [0]:

# ============================================================
# AUDIT - PORTFOLIO EXCEPTION EXECUTION SUMMARY
# Daily snapshot for management dashboards.
# 
# This writes two audit tables:
#   1. portfolio_exception_execution_summary - overall exception counts
#   2. business_area_exception_summary - exceptions by business area
# ============================================================

end_time         = datetime.now()
duration_seconds = int((end_time - start_time).total_seconds())

# Calculate summary statistics from enriched_df
if enriched_df.count() > 0:
    summary = enriched_df.groupBy().agg(
        F.count("*").alias("total_exceptions"),
        F.sum(F.when(F.col("severity_code") == "CRITICAL", 1).otherwise(0)).alias("critical_exceptions"),
        F.sum(F.when(F.col("severity_code") == "HIGH", 1).otherwise(0)).alias("high_exceptions"),
        F.sum(F.when(F.col("severity_code") == "MEDIUM", 1).otherwise(0)).alias("medium_exceptions"),
        F.sum(F.when(F.col("severity_code") == "LOW", 1).otherwise(0)).alias("low_exceptions"),
        F.sum(F.when(F.col("escalation_required") == True, 1).otherwise(0)).alias("escalated_exceptions")
    ).collect()[0]
    
    tot  = summary.total_exceptions or 0
    crit = summary.critical_exceptions or 0
    high = summary.high_exceptions or 0
    med  = summary.medium_exceptions or 0
    low  = summary.low_exceptions or 0
    esc  = summary.escalated_exceptions or 0
else:
    tot = crit = high = med = low = esc = 0

# 15.1 Write to Portfolio Exception Execution Summary
spark.sql(f"""
    INSERT INTO retailbank_dev.audit.portfolio_exception_execution_summary
    (business_date, execution_id, procedure_name, total_exceptions, 
     critical_exceptions, high_exceptions, medium_exceptions, low_exceptions, 
     escalated_exceptions, execution_seconds, created_date)
    VALUES
    ('{business_date}', '{execution_id}', '{procedure_name}', 
     {tot}, {crit}, {high}, {med}, {low}, {esc}, 
     {duration_seconds}, '{end_time.strftime("%Y-%m-%d %H:%M:%S")}')
""")

# 15.2 Write to Business Area Exception Summary
if enriched_df.count() > 0:
    biz_summary = enriched_df.groupBy("business_area").agg(
        F.count("*").alias("exception_count"),
        F.sum(F.when(F.col("severity_code") == "CRITICAL", 1).otherwise(0)).alias("critical_count")
    )
    
    biz_summary = biz_summary.withColumn("business_date", F.lit(business_date).cast("date")) \
        .withColumn("created_date", F.current_timestamp())
    
    biz_summary.write.format("delta").mode("append").saveAsTable("retailbank_dev.audit.business_area_exception_summary")

if debug:
    print("Audit summaries written.")
    print(f"  Total exceptions        : {tot}")
    print(f"  Critical exceptions     : {crit}")
    print(f"  High exceptions         : {high}")
    print(f"  Medium exceptions       : {med}")
    print(f"  Low exceptions          : {low}")
    print(f"  Escalated exceptions    : {esc}")

Audit summaries written.
  Total exceptions        : 6
  Critical exceptions     : 3
  High exceptions         : 3
  Medium exceptions       : 0
  Low exceptions          : 0
  Escalated exceptions    : 6


In [0]:
# Databricks notebook source
# ============================================================
# FINAL AUDIT UPDATE
# 
# Updates the ETL execution log with final status and statistics.
# This is the same pattern used in nb_01, nb_02, and nb_03.
# ============================================================

status  = "SUCCESS"
message = (
    f"Portfolio Exception Processing Completed. "
    f"Exceptions Identified: {rows_inserted}, "
    f"Records Read: {rows_read}, "
    f"Duplicates Removed: {duplicate_exceptions_removed}, "
    f"Processing Time: {duration_seconds} seconds."
)

write_audit_end(status, message)

if debug:
    print("Final audit log updated.")

Final audit log updated.


In [0]:
# Databricks notebook source
# ============================================================
# SUMMARY OUTPUT
# 
# Displays the final summary of the portfolio exception load.
# This matches the output style of nb_03.
# ============================================================

if debug:
    print("=" * 50)
    print("PORTFOLIO EXCEPTION LOAD SUMMARY")
    print("=" * 50)
    print(f"Records Read        : {rows_read}")
    print(f"Exceptions Loaded   : {rows_inserted}")
    print(f"Duplicates Removed  : {duplicate_exceptions_removed}")
    print(f"Execution Time      : {duration_seconds} seconds")
    print(f"Status              : {status}")
    print("=" * 50)
    
    # Show exception distribution by severity
    if enriched_df.count() > 0:
        print("\nException Distribution by Severity:")
        enriched_df.groupBy("severity_code").count().orderBy("severity_code").show()
        
        print("\nException Distribution by Rule:")
        enriched_df.groupBy("rule_code", "exception_category").count().orderBy("rule_code").show()
        
        print("\nSample Exceptions (first 5):")
        enriched_df.select(
            "source_system_code", 
            "source_account_number", 
            "rule_code", 
            "severity_code", 
            "exception_description"
        ).show(5, truncate=False)

print("nb_04_PortfolioExceptions completed successfully.")

PORTFOLIO EXCEPTION LOAD SUMMARY
Records Read        : 19
Exceptions Loaded   : 6
Duplicates Removed  : 0
Execution Time      : 33 seconds
Status              : SUCCESS

Exception Distribution by Severity:
+-------------+-----+
|severity_code|count|
+-------------+-----+
|     CRITICAL|    3|
|         HIGH|    3|
+-------------+-----+


Exception Distribution by Rule:
+---------+--------------------+-----+
|rule_code|  exception_category|count|
+---------+--------------------+-----+
|    DQ001| Customer Validation|    3|
|    DQ004|Financial Validation|    3|
+---------+--------------------+-----+


Sample Exceptions (first 5):
+------------------+---------------------+---------+-------------+-------------------------------------+
|source_system_code|source_account_number|rule_code|severity_code|exception_description                |
+------------------+---------------------+---------+-------------+-------------------------------------+
|CORE_BANKING      |CB10004              |DQ004 